# Integração Pollen AI Atlas × pipeline de vistas (Júlio) — v2, folds da tese
**Pergunta:** um foundation model (DINOv2 / Pollen AI Atlas) classifica o CPD1 melhor que a DenseNet201 da tese, no teste completo e nas vistas equatorial e polar?

**Protocolo igual ao da tese:** para cada um dos **10 folds exatos do Júlio**, o modelo aprende com todas as imagens de treino e é avaliado no teste, que depois é separado em vista equatorial e polar. A lista de folds e vistas vem do repositório `Pollen-Classification-CBAM` (`results/phase3/CPD1_TEST_VIEW`).

**Método (linear probe):** o ViT fica congelado e gera um vetor de 384 números por imagem; uma regressão logística aprende a classificar esses vetores.

**Antes de rodar:** Ambiente de execução → Alterar tipo de ambiente → **GPU T4**.

In [1]:
!pip install -q timm openpyxl
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Copia a base para o disco do Colab (ler direto do Drive é muito lento)
!cp /content/drive/MyDrive/IC_Polen/CPD1_Cr_Rs_400.zip /content/
!unzip -q -n /content/CPD1_Cr_Rs_400.zip -d /content/BD/
!ls /content/BD/CPD1_Cr_Rs_400

balance_log_2026-11-10.csv  csv  Test  Train


## 1. Configuração — ajuste só esta célula
Comece com `FOLDS = [1]` para testar; quando funcionar, troque para `list(range(1, 11))`.

In [3]:
DRIVE = "/content/drive/MyDrive/IC_Polen"
PASTA_IMAGENS = "/content/BD/CPD1_Cr_Rs_400/Test"        # contém todas as imagens originais (k1..k10)
MAPA = f"{DRIVE}/vistas_folds_julio_CPD1.csv"             # folds e vistas da tese
BASELINE = f"{DRIVE}/baseline_DenseNet201_tese_por_fold.xlsx"
CKPT = f"{DRIVE}/weights_vit_small_lvd_20250620_0312.pth"
OUT  = f"{DRIVE}/resultados_atlas_v2"
FOLDS = [1]                 # teste rápido; depois: list(range(1, 11))
IMG_SIZE = 518
BATCH = 64

In [4]:
import os, pandas as pd
EXTS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
caminho = {}                                    # "classe/arquivo" -> caminho no disco
for raiz, _, arqs in os.walk(PASTA_IMAGENS):
    for a in arqs:
        if a.lower().endswith(EXTS) and not a.lower().startswith("aug_"):
            caminho[f"{os.path.basename(raiz)}/{a}"] = os.path.join(raiz, a)
mapa = pd.read_csv(MAPA)
os.makedirs(OUT, exist_ok=True)
faltando = sorted(set(mapa.file) - set(caminho))
print(f"Imagens encontradas: {len(caminho)} | na lista da tese: {len(mapa)} | faltando: {len(faltando)}")
if faltando: print("Exemplos faltando:", faltando[:5])
print(mapa.groupby(['k', 'vista']).size().unstack())
print("Checkpoint:", "OK" if os.path.isfile(CKPT) else "NÃO ENCONTRADO")
print("Baseline:", "OK" if os.path.isfile(BASELINE) else "NÃO ENCONTRADO")

Imagens encontradas: 4025 | na lista da tese: 4025 | faltando: 0
vista  EQUATORIAL  POLAR
k                       
1             210    193
2             219    184
3             209    194
4             213    190
5             210    193
6             216    186
7             210    192
8             201    201
9             216    186
10            206    196
Checkpoint: OK
Baseline: OK


## 2. Carregar os modelos e checar se o backbone do Atlas foi ajustado com pólen
Se a diferença der ~0, os pesos do Atlas são iguais ao DINOv2 original e os dois modelos vão dar o mesmo resultado.

In [5]:
import numpy as np, torch, timm, pickle
from PIL import Image
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device, torch.cuda.get_device_name(0) if device == "cuda" else "(sem GPU!)")

def criar_vit(pretrained):
    return timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=pretrained,
                             img_size=IMG_SIZE, init_values=1e-5, num_classes=0)

atlas = criar_vit(False)
sd = torch.load(CKPT, map_location="cpu")
sd = {k: v for k, v in sd.items() if not k.startswith("head")}
r = atlas.load_state_dict(sd, strict=False)
print("Chaves faltando:", r.missing_keys, "| sobrando:", r.unexpected_keys)
assert not r.missing_keys, "O checkpoint não bateu com o modelo"

dino = criar_vit(True)
ref = dino.state_dict()
diff = max((atlas.state_dict()[k] - ref[k]).abs().max().item() for k in ref)
print(f"Maior diferença de peso Atlas x DINOv2 original: {diff:.2e}")
print("=> backbone IGUAL ao DINOv2 original" if diff < 1e-6 else "=> backbone FOI ajustado com pólen")
MODELOS = {"Atlas": atlas.eval().to(device), "DINOv2": dino.eval().to(device)}

Dispositivo: cuda Tesla T4
Chaves faltando: [] | sobrando: []


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Maior diferença de peso Atlas x DINOv2 original: 2.27e-03
=> backbone FOI ajustado com pólen


## 3. Extrair os vetores de todas as imagens (uma vez por modelo)
Fica salvo no Drive: se o Colab desconectar, ele continua de onde parou.

In [6]:
tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
os.makedirs(OUT, exist_ok=True)
chaves = list(mapa.file)

@torch.no_grad()
def extrair(nome, modelo):
    arq = f"{OUT}/vetores_{nome}.pkl"
    cache = pickle.load(open(arq, "rb")) if os.path.exists(arq) else {}
    faltam = [c for c in chaves if c not in cache]
    for i in range(0, len(faltam), BATCH):
        lote = faltam[i:i + BATCH]
        x = torch.stack([tf(Image.open(caminho[c]).convert("RGB")) for c in lote]).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            z = modelo(x).float().cpu().numpy()
        for c, v in zip(lote, z): cache[c] = v
        if (i // BATCH) % 10 == 0:
            pickle.dump(cache, open(arq, "wb")); print(f"{nome}: {len(cache)}/{len(chaves)}")
    pickle.dump(cache, open(arq, "wb"))
    return cache

VETORES = {nome: extrair(nome, m) for nome, m in MODELOS.items()}
print("Pronto.")

Atlas: 64/4025
Atlas: 704/4025
Atlas: 1344/4025
Atlas: 1984/4025
Atlas: 2624/4025
Atlas: 3264/4025
Atlas: 3904/4025
DINOv2: 64/4025
DINOv2: 704/4025
DINOv2: 1344/4025
DINOv2: 1984/4025
DINOv2: 2624/4025
DINOv2: 3264/4025
DINOv2: 3904/4025
Pronto.


## 4. Treinar e avaliar nos folds da tese

In [10]:
FOLDS = list(range(1, 11))
print(FOLDS)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, cohen_kappa_score

def metricas(yt, yp):
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(yt, yp), "precision": p, "recall": r, "fscore": f,
            "kappa": cohen_kappa_score(yt, yp)}

classes = sorted(mapa.classe.unique()); idx = {c: i for i, c in enumerate(classes)}
linhas, preds = [], []
for nome, vet in VETORES.items():
    for k in FOLDS:
        tr, te = mapa[mapa.k != k], mapa[mapa.k == k]
        Xtr = np.stack([vet[c] for c in tr.file]); ytr = tr.classe.map(idx).values
        Xte = np.stack([vet[c] for c in te.file]); yte = te.classe.map(idx).values
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(Xtr, ytr)
        yp = clf.predict(Xte)
        for vista in ["TODAS", "EQUATORIAL", "POLAR"]:
            sel = np.ones(len(te), bool) if vista == "TODAS" else (te.vista.values == vista)
            linhas.append({"Modelo": nome, "Vista": vista, "k": k, **metricas(yte[sel], yp[sel])})
        preds.append(te.assign(Modelo=nome, predicted_label=[classes[i] for i in yp]))
        l = linhas[-3:]
        print(f"{nome:7} k{k:<2}  todas={l[0]['accuracy']:.4f}  equatorial={l[1]['accuracy']:.4f}  polar={l[2]['accuracy']:.4f}")
res = pd.DataFrame(linhas); res.to_csv(f"{OUT}/metricas_por_fold.csv", index=False)
pd.concat(preds).assign(status=lambda d: np.where(d.classe == d.predicted_label, "Correct", "Incorrect")) \
  .to_csv(f"{OUT}/predicoes.csv", index=False)

Atlas   k1   todas=0.9653  equatorial=0.9619  polar=0.9689
Atlas   k2   todas=0.9851  equatorial=0.9726  polar=1.0000
Atlas   k3   todas=0.9777  equatorial=0.9713  polar=0.9845
Atlas   k4   todas=0.9752  equatorial=0.9812  polar=0.9684
Atlas   k5   todas=0.9727  equatorial=0.9667  polar=0.9793
Atlas   k6   todas=0.9677  equatorial=0.9630  polar=0.9731
Atlas   k7   todas=0.9602  equatorial=0.9619  polar=0.9583
Atlas   k8   todas=0.9627  equatorial=0.9552  polar=0.9701
Atlas   k9   todas=0.9627  equatorial=0.9583  polar=0.9677
Atlas   k10  todas=0.9627  equatorial=0.9515  polar=0.9745
DINOv2  k1   todas=0.9727  equatorial=0.9667  polar=0.9793
DINOv2  k2   todas=0.9901  equatorial=0.9909  polar=0.9891
DINOv2  k3   todas=0.9677  equatorial=0.9474  polar=0.9897
DINOv2  k4   todas=0.9851  equatorial=0.9812  polar=0.9895
DINOv2  k5   todas=0.9628  equatorial=0.9524  polar=0.9741
DINOv2  k6   todas=0.9751  equatorial=0.9630  polar=0.9892
DINOv2  k7   todas=0.9577  equatorial=0.9524  polar=0.96

## 5. Comparar com a DenseNet201 da tese (mesmos folds)

In [12]:
base = []
for vista in ["EQUATORIAL", "POLAR"]:
    d = pd.read_excel(BASELINE, sheet_name=vista)
    d = d[pd.to_numeric(d.k, errors="coerce").notna()].astype({"k": int})
    base.append(d.assign(Modelo="DenseNet201 (tese)", Vista=vista))
base = pd.concat(base)[["Modelo", "Vista", "k", "accuracy", "precision", "recall", "fscore", "kappa"]]
base = base[base.k.isin(FOLDS)]
tudo = pd.concat([res, base])
resumo = (tudo.groupby(["Vista", "Modelo"])[["accuracy", "precision", "recall", "fscore", "kappa"]]
            .agg(["mean", "std"]) * 100).round(2)
display(resumo)
resumo.to_excel(f"{OUT}/resumo_comparacao.xlsx")

accuracy       precision       recall        \
                                  mean   std      mean   std   mean   std   
Vista      Modelo                                                           
EQUATORIAL Atlas                 96.44  0.89     96.91  0.90  96.44  0.89   
           DINOv2                96.44  1.42     96.92  1.45  96.44  1.42   
           DenseNet201 (tese)    98.94  1.17     99.15  0.94  98.94  1.17   
POLAR      Atlas                 97.45  1.14     97.85  1.09  97.45  1.14   
           DINOv2                97.92  1.03     98.36  1.06  97.92  1.03   
           DenseNet201 (tese)    99.18  0.79     99.24  0.86  99.18  0.79   
TODAS      Atlas                 96.92  0.82     97.08  0.76  96.92  0.82   
           DINOv2                97.14  1.06     97.31  1.02  97.14  1.06   

                              fscore        kappa        
                                mean   std   mean   std  
Vista      Modelo                                        
EQUATORIAL Atlas               96.48  0.92  95.66  1.10  
           DINOv2              96.47  1.49  95.66  1.74  
           DenseNet201 (tese)  98.98  1.14  98.70  1.44  
POLAR      Atlas               97.51  1.12  96.75  1.46  
           DINOv2              98.03  1.07  97.35  1.30  
           DenseNet201 (tese)  99.13  0.93  98.94  1.01  
TODAS      Atlas               96.91  0.81  96.56  0.91  
           DINOv2              97.14  1.06  96.81  1.19

## 6. Teste de Wilcoxon pareado por fold (precisa dos 10 folds)

In [13]:
from scipy.stats import wilcoxon
for nome in MODELOS:
    for vista in ["EQUATORIAL", "POLAR"]:
        a = res[(res.Modelo == nome) & (res.Vista == vista)].sort_values("k").accuracy.values
        b = base[base.Vista == vista].sort_values("k").accuracy.values
        if len(a) < 5:
            print(f"{nome} x DenseNet201 ({vista}): rode os 10 folds para o teste"); continue
        dif = a - b
        try:
            p = wilcoxon(a, b, zero_method="zsplit").pvalue
            print(f"{nome} x DenseNet201 ({vista}): diferença média {dif.mean()*100:+.2f} pp | p = {p:.4f}"
                  f" -> {'significativo' if p < 0.05 else 'não significativo'}")
        except ValueError as e:
            print(f"{nome} x DenseNet201 ({vista}): não foi possível calcular ({e})")

Atlas x DenseNet201 (EQUATORIAL): diferença média -2.50 pp | p = 0.0020 -> significativo
Atlas x DenseNet201 (POLAR): diferença média -1.73 pp | p = 0.0039 -> significativo
DINOv2 x DenseNet201 (EQUATORIAL): diferença média -2.50 pp | p = 0.0039 -> significativo
DINOv2 x DenseNet201 (POLAR): diferença média -1.26 pp | p = 0.0020 -> significativo
